In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

<b><font color="red" size="10">ch15. 데이터 배이스 연동</font></b>

# 1절. SQLite 데이터 배이스 연결
- SQLite 데이터 배이스는 별도의 DBMS없이 SQL을 이용해서 DB액세스할 수 있도록 간단한 디스크기반 DB제공
- C라이브러리
- SQLite는 프로토 타입을 만들때 사용
- 프로젝트 단계 : 분석   ->   설계 ->   구현 ->   테스트 ->   고객에게 배포 ->   유지보수 
            -     프로토타입(SQLite)  시제품 (구현후 반양산직전) 완제품
                                       (Oracle,MySQL,MariaDB,PostgreSQL,MSSQL,아마존aurorDB, ...)  
- [DB Browers for SQLite](https://sqlitebrowser.org/)에서 "DB Browser for SQLite - .zip (no installer) for 64-bit Windows" 다운로드후 압축 풀기  

## 1.1 SQLite browser 설치 및sqlite3패키지 load

In [7]:
import sqlite3
sqlite3.sqlite_version

'3.40.1'

In [6]:
import pandas as pd
pd.__version__

'1.5.3'

## 1.2 데이터 배이스 연결
- 데이터배이스 연결 객체 -> 커서객체(SQL전송 및 결과 받는객체) ->원하는 로직 수행->커서객체 해제->DB연결객체 해제(close)
- SQLite로 DB 연결객체 생성시,DB파일이 있으면 연결, DB파일이 없으면 빈 DB파일 생성 

In [17]:
# DB연결(여기서 에러가 날 경우 VC_redist.x64.exe 설치)
conn = sqlite3.connect('data/ch15_example.db')
conn

In [18]:
# 커서 객체 생성 : 커서는 SQL문 실행시키고, 결고를 받는 객체
cursor = conn.cursor()
cursor

In [19]:
cursor.execute('''
    CREATE TABLE MEMBER(
        NAME TEXT,
        AGE INT,
        EMAIL TEXT 
    )
''')

OperationalError: database is locked

In [ ]:
cursor.execute('DROP TABLE MEMBER')

In [22]:
sql = 'INSERT INTO MEMBER VALUES (\'마길동\',25,\'H@g.com\')'
cursor.execute(sql) # sql전송 
print('insert,update,delete문의 수행 결과 행수:', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신길동',30,'H@g.com\')"
cursor.execute(sql)
print('수행결과 행수:',cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('신림동',35,'H@g.com\')"
cursor.execute(sql)
print('수행결과 행수:',cursor.rowcount)

insert,update,delete문의 수행 결과 행수: 1
수행결과 행수: 1
수행결과 행수: 1


In [23]:
conn.commit()# 反.conn.rollback()DML에서만 commit이나 rollback

In [24]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE") #SELECT sql문 전송 

In [25]:
# INSERT, UPDATE, DELETE문 실행결과 : cursor.rowcount
# SELECT문 실행결과를 받는 함수들 
    #fetchone() : 결과를 환행씩 받을때 (튜플)
    #fetchall() : 결과를 모두 받을떄 (튜플 list)
    #fetchmany(n) : 결과를 n행 받을때(튜플 list)
cursor.fetchall()

[('홍길동', 25, 'H@g.com'),
 ('마길동', 25, 'H@g.com'),
 ('마길동', 25, 'H@g.com'),
 ('신길동 ', 30, 'H@g.com'),
 ('신길동 ', 30, 'H@g.com'),
 ('신림동 ', 35, 'H@g.com')]

In [26]:
cursor.fetchall() # 한번 소요된 cursor 객체는 다시 fetch할수없음

[]

In [27]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchall()
members

[('홍길동', 25, 'H@g.com'),
 ('마길동', 25, 'H@g.com'),
 ('마길동', 25, 'H@g.com'),
 ('신길동 ', 30, 'H@g.com'),
 ('신길동 ', 30, 'H@g.com'),
 ('신림동 ', 35, 'H@g.com')]

In [29]:
# 한줄씩 읽어오기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    member = cursor.fetchone()# SQL문 수행결과를 한줄 가져오기
    if member is None:
        break
    members.append({'name':member[0],'age':[1],'email':[2]})
members

[{'name': '홍길동', 'age': [1], 'email': [2]},
 {'name': '마길동', 'age': [1], 'email': [2]},
 {'name': '마길동', 'age': [1], 'email': [2]},
 {'name': '신길동 ', 'age': [1], 'email': [2]},
 {'name': '신길동 ', 'age': [1], 'email': [2]},
 {'name': '신림동 ', 'age': [1], 'email': [2]}]

In [32]:
class Member:
    'Member 태이블의 내용을 받은 객체 타입'
    def __init__(self,name,age,email):
        self.name = name
        self.age = age
        self.email=email
    def __str__(self):
        return "{}\t{}\t{}".format(self.name, self.age, self.email)
m=Member('홍길동',25,'h@h.com')
print(m)

홍길동	25	h@h.com


In [38]:
# 한줄씩 읽어서 객체 list에 append
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
member = []
while True:
    dbmember = cursor.fetchone()
    if dbmember is None:
        break
    member=Member(*dbmember)
    members.append(member)
for mem in members:
    print(mem)

{'name': '홍길동', 'age': [1], 'email': [2]}
{'name': '마길동', 'age': [1], 'email': [2]}
{'name': '마길동', 'age': [1], 'email': [2]}
{'name': '신길동 ', 'age': [1], 'email': [2]}
{'name': '신길동 ', 'age': [1], 'email': [2]}
{'name': '신림동 ', 'age': [1], 'email': [2]}
홍길동	25	H@g.com
마길동	25	H@g.com
마길동	25	H@g.com
신길동 	30	H@g.com
신길동 	30	H@g.com
신림동 	35	H@g.com


In [40]:
# 최상위 n행 읽어오기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchmany(2)
members

[('홍길동', 25, 'H@g.com'), ('마길동', 25, 'H@g.com')]

In [41]:
import pandas as pd
pd.DataFrame(members)

,0,1,2
0,홍길동,25,H@g.com
1,마길동,25,H@g.com


In [ ]:
cursor.close()
conn.close()

## 1.3 SQL구문에 파라미터 사용하기

- qmark(DB에따라 불가한경우가 있음) 
- named(추천)

In [51]:
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN('홍길동','신길동')")
cursor.fetchall()

OperationalError: no such column: NAME

In [52]:
# 파라미터 사용하기 : qmark이용
name1 = input('검색할 이름:')
name2 = input('검색할 이름:')
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN(:name1,:name2)",{'name1':name1, 'name2':name2})
cursor.fetchall()

검색할 이름:홍길동
검샐할 이름:신림동


OperationalError: no such column: NAME

In [45]:
#파라미터 이용하기 : named방법
sql = "INSERT INTO MEMBER VALUSE (:name,:age, :email)"
name = input('회원가입할 이름은?')
try:
    age = int(input('나이는?(꼭 숫자로)'))
except:
    print('유효하지 않은 나이를 입력할경우 1세로 초기화합니다.')
    age = 1
email = input('이메일은?')
newMember = ("INSERT INTO MEMBER VALUES(:name, :age, :email)",
            {'name':name,'age':age,'email':email})
print('수행 결과 행수 : ', cursor.rowcount)

회원가입할 이름은?윤길동
나이는?(꼭 숫자로)이십팔세
유효하지 않은 나이를 입력할경우 1세로 초기화합니다.
이메일은?aaa@ff.com
수행 결과 행수 :  -1


In [53]:
cursor.close()
conn.close()

# 2절. 오라클 데이터 배이스 연결

- pip install cx_oracle(11g까지), pip install oracledb(12버전부터)

In [54]:
import cx_Oracle
cx_Oracle.__version__

'8.3.0'

In [55]:
# conn 얻어오는 방법 1
oracle_dsn = cx_Oracle.makedsn(host="localhost",port=1521, sid='xe')
# conn = cx_Oracle.connect(user='scott',password='tiger',dsn=oracle_dsn)
conn = cx_Oracle.connect('scott','tiger',oracle_dsn)
conn.close()

In [56]:
#conn 얻어오는 방법 2(여기서 에러가 날 경우 VC_redist.x64.exe 설치)
conn = cx_Oracle.connect('scott','tiger','localhost:1521/xe')
conn

<cx_Oracle.Connection to scott@localhost:1521/xe>

In [58]:
# cursor 객체 생성하고 sql문 전송 & 결과 받기
cursor = conn.cursor()
sql = "SELECT EMPNO NO, ENAME, JOB, MGR, HIREDATE, SAL, COMM, DEPTNO FROM EMP"
cursor.execute(sql)
emps = cursor.fetchall()


In [59]:
for emp in emps:
    print(emp)

(7369, 'SMITH', 'CLERK', 7902, datetime.datetime(1980, 12, 17, 0, 0), 800.0, None, 20)
(7499, 'ALLEN', 'SALESMAN', 7698, datetime.datetime(1981, 2, 20, 0, 0), 1600.0, 300.0, 30)
(7521, 'WARD', 'SALESMAN', 7698, datetime.datetime(1981, 2, 22, 0, 0), 1250.0, 500.0, 30)
(7566, 'JONES', 'MANAGER', 7839, datetime.datetime(1981, 4, 2, 0, 0), 2975.0, None, 20)
(7654, 'MARTIN', 'SALESMAN', 7698, datetime.datetime(1981, 9, 28, 0, 0), 1250.0, 1400.0, 30)
(7698, 'BLAKE', 'MANAGER', 7839, datetime.datetime(1981, 5, 1, 0, 0), 2850.0, None, 30)
(7782, 'CLARK', 'MANAGER', 7839, datetime.datetime(1981, 6, 9, 0, 0), 2450.0, None, 10)
(7788, 'SCOTT', 'ANALYST', 7566, datetime.datetime(1982, 12, 9, 0, 0), 3000.0, None, 20)
(7839, 'KING', 'PRESIDENT', None, datetime.datetime(1981, 11, 17, 0, 0), 5000.0, None, 10)
(7844, 'TURNER', 'SALESMAN', 7698, datetime.datetime(1981, 9, 8, 0, 0), 1500.0, 0.0, 30)
(7876, 'ADAMS', 'CLERK', 7788, datetime.datetime(1983, 1, 12, 0, 0), 1100.0, None, 20)
(7900, 'JAMES', 'CL

In [ ]:
# import pandas as pd
# emp_df = pd.DataFrame(emps)
# emp_df

In [60]:
cursor.description

[('NO', <cx_Oracle.DbType DB_TYPE_NUMBER>, 5, None, 4, 0, 0),
 ('ENAME', <cx_Oracle.DbType DB_TYPE_VARCHAR>, 10, 10, None, None, 1),
 ('JOB', <cx_Oracle.DbType DB_TYPE_VARCHAR>, 9, 9, None, None, 1),
 ('MGR', <cx_Oracle.DbType DB_TYPE_NUMBER>, 5, None, 4, 0, 1),
 ('HIREDATE', <cx_Oracle.DbType DB_TYPE_DATE>, 23, None, None, None, 1),
 ('SAL', <cx_Oracle.DbType DB_TYPE_NUMBER>, 11, None, 7, 2, 1),
 ('COMM', <cx_Oracle.DbType DB_TYPE_NUMBER>, 11, None, 7, 2, 1),
 ('DEPTNO', <cx_Oracle.DbType DB_TYPE_NUMBER>, 3, None, 2, 0, 1)]

In [61]:
[descript[0] for descript in cursor.description]

['NO', 'ENAME', 'JOB', 'MGR', 'HIREDATE', 'SAL', 'COMM', 'DEPTNO']

In [64]:
import pandas as pd
emp_df = pd.DataFrame(emps,
                     columns=[descript[0] for descript in cursor.description])
emp_df

,NO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7369,SMITH,CLERK,7902.0,1980-12-17,800.0,NaN,20
1,7499,ALLEN,SALESMAN,7698.0,1981-02-20,1600.0,300.0,30
2,7521,WARD,SALESMAN,7698.0,1981-02-22,1250.0,500.0,30
3,7566,JONES,MANAGER,7839.0,1981-04-02,2975.0,NaN,20
4,7654,MARTIN,SALESMAN,7698.0,1981-09-28,1250.0,1400.0,30
5,7698,BLAKE,MANAGER,7839.0,1981-05-01,2850.0,NaN,30
6,7782,CLARK,MANAGER,7839.0,1981-06-09,2450.0,NaN,10
7,7788,SCOTT,ANALYST,7566.0,1982-12-09,3000.0,NaN,20
8,7839,KING,PRESIDENT,NaN,1981-11-17,5000.0,NaN,10
9,7844,TURNER,SALESMAN,7698.0,1981-09-08,1500.0,0.0,30


In [74]:
# 사용자로부터 검색할 이름을 받아 해당 데이타 출력
sql = "SELECT * FROM EMP WHERE ENAME=(:ename)"
ename = input('검색할 이름?').upper()
cursor.execute(sql,{'ename':ename})
emp = cursor.fetchone() #결과가 있으면 해당 데이터를 튜플로, 결과가 없으면 None으로
if emp:
    columns = [descript[0] for descript in cursor.description]
    df = pd.DataFrame([emp],columns=columns)
    display(df)
else:
    print('해당이름이 없습니다.')

검색할 이름?smith


,EMPNO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7369,SMITH,CLERK,7902,1980-12-17,800.0,None,20


In [76]:
for col, data in zip(columns,emp):
    print("{}:{}".format(col, data if data is not None else '-'))

EMPNO:7369
ENAME:SMITH
JOB:CLERK
MGR:7902
HIREDATE:1980-12-17 00:00:00
SAL:800.0
COMM:-
DEPTNO:20


In [77]:
cursor.close()
conn.close()

# 3절. MySQL 연결

|라이브러리 | 특징 |
|:--|:--|
|**mysql-connector-python**|MySQL 공식 커넥터. python으로 구현|
|**PyMySQL**||커뮤니티에서 만든 

In [78]:
%pip install pyMySQL

     ---------------------------------------- 45.7/45.7 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pymysql
conn = pymysql.connect(
    # host,port,username,password,database,charset,autocommit
)